In [ ]:
# Importação das bibliotecas essenciais

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from kagglehub import KaggleDatasetAdapter
import json
import numpy as np
import re
# Bibliotecas adicionadas para melhor EDA e análise estatística
from scipy.stats import shapiro
import plotly.express as px
from pathlib import Path


# Seção: Business Understanding
print("=== Business Understanding ===")
print("Objetivo: Avaliar discoverability de perfis/posts no LinkedIn como % eficiência (0-100%), simulando algoritmo para otimizar visibilidade profissional.")
print("Motivação em 2026: Com >1B usuários, LinkedIn usa IA para recomendações; modelo ajuda a reduzir vieses (ex: perfis sub-representados) e melhora networking.")
print("Exemplos: Perfis com skills relevantes +20% score; posts virais vs. genéricos.")
print("Implicações: Éticas (privacidade dados); Sociais (igualdade em recrutamento); Limitações: Proxy baseado em dados públicos, não algoritmo real.")
print("Edge Cases: Perfis novos (baixa rede) ou posts com spam (queda em score).")

In [ ]:
# Carregar datasets diretamente em DataFrames do pandas através do Kaggle

dataset_perfis = "likithagedipudi/linkedin-compatibility-dataset-50k-profiles"

# Dataframe para perfis do LinkedIn
df_profiles = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    dataset_perfis,
    "profiles.csv",
)

# Dataframe para pares de compatibilidade
df_pairs = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    dataset_perfis,
    "compatibility_pairs.csv",
)

dataset_influencers = "shreyasajal/linkedin-influencers-data"

# Dataframe para dados de influenciadores do LinkedIn
df_influencers = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    dataset_influencers,
    "influencers_data.csv",
)

In [ ]:
# Ver informações dos dataframes
print("Informação de Perfis :")
display(df_profiles.info())
display(df_profiles.describe())
print("="*70)

print("Informação de Pairs :")
display(df_pairs.info())
display(df_pairs.describe())
print("="*70)

print("Informação de Influencers :")
display(df_influencers.info())
display(df_influencers.describe())

In [ ]:
# Limpar dataframe de perfis
def parse_json_safely(value):
    if pd.isna(value) or value == "":
        return []  # Return empty list so .apply(len) works later
    if isinstance(value, str):
        try:
            return json.loads(value)
        except (json.JSONDecodeError, TypeError):
            return []
    return value

cols_to_parse = ["skills", "experience", "education", "goals", "needs", "can_offer"]

for col in cols_to_parse:
    if col in df_profiles.columns:
        df_profiles[col] = df_profiles[col].apply(parse_json_safely)

# Criar features de contagem
df_profiles["num_skills"] = df_profiles["skills"].apply(len)
df_profiles["num_experience"] = df_profiles["experience"].apply(len)
df_profiles["num_education"] = df_profiles["education"].apply(len)
df_profiles["about_length"] = df_profiles["about"].apply(lambda x: len(x) if isinstance(x, str) else 0)
df_profiles["headline_length"] = df_profiles["headline"].apply(lambda x: len(x) if isinstance(x, str) else 0)

# Remover duplicados
df_profiles = df_profiles.drop_duplicates(subset=["profile_id"], keep="first").reset_index(drop=True)

In [ ]:
# Limpar dataframe de pares de compatibilidade

# Remover coluna inútil
df_pairs = df_pairs.drop(columns=["skill_complementarity_score"], errors="ignore")

# Garantir integridade dos pares
valid_profile_ids = set(df_profiles["profile_id"])
df_pairs = df_pairs[
    df_pairs["profile_a_id"].isin(valid_profile_ids) & 
    df_pairs["profile_b_id"].isin(valid_profile_ids)
].reset_index(drop=True)

print(f"Linhas após filtro de IDs válidos: {len(df_pairs):,}")

In [ ]:
# =============================================================================
# SCORE DE "DESCOBERTA" DO PERFIL (0 a 100)
# =============================================================================

avg_compat = df_pairs.groupby('profile_a_id')['compatibility_score'].mean().rename('avg_compatibility_score')

if 'avg_compatibility_score' not in df_profiles.columns:
    df_profiles = df_profiles.merge(
        avg_compat, 
        left_on='profile_id', 
        right_index=True, 
        how='left'
    )
    print("Coluna 'avg_compatibility_score' adicionada com sucesso.")
else:
    print("Coluna 'avg_compatibility_score' já existe → merge ignorado.")

df_profiles['avg_compatibility_score'] = df_profiles['avg_compatibility_score'].fillna(0)

if 'avg_compatibility_score' in df_profiles.columns:
    min_compat = df_profiles["avg_compatibility_score"].min()
    max_compat = df_profiles["avg_compatibility_score"].max()

    if max_compat > min_compat:
        df_profiles["profile_discoverability_score"] = 100 * (
            (df_profiles["avg_compatibility_score"] - min_compat) / (max_compat - min_compat)
        )
    else:
        df_profiles["profile_discoverability_score"] = 0.0

In [ ]:
# Limpar dataframe de influencers

df_influencers = df_influencers.drop(columns=["views", "votes", "connections"], errors="ignore")

if df_influencers.index.name == "serial_number":
    print("serial_number já é o índice.")
else:
    rename_map = {
        "Unnamed: 0": "serial_number",
        "unnamed: 0": "serial_number",
        "index": "serial_number",
        "Index": "serial_number",
    }
    df_influencers = df_influencers.rename(columns=rename_map)

    if "serial_number" in df_influencers.columns:
        df_influencers = df_influencers.set_index("serial_number", verify_integrity=False)
        print("serial_number definido como índice.")
    else:
        print("Não foi encontrada coluna para serial_number.")

df_influencers["location"] = df_influencers["location"].fillna("Unknown")
df_influencers["followers"] = df_influencers["followers"].astype("Int64").fillna(0)
df_influencers["content"] = df_influencers["content"].fillna("Empty Post")
df_influencers["media_type"] = df_influencers["media_type"].fillna("Unknown")

def time_spent_to_days(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().lower()
    if text in {"just now", "now"}:
        return 0
    m = re.match(r"^\s*(\d+)\s+([a-z]+)\s+ago\s*$", text)
    if not m:
        return pd.NA
    qty = int(m.group(1))
    unit = m.group(2)
    factors = {
        "minute": 0, "minutes": 0, "hour": 0, "hours": 0,
        "day": 1, "days": 1, "week": 7, "weeks": 7,
        "month": 30, "months": 30, "year": 365, "years": 365,
    }
    if unit not in factors:
        return pd.NA
    return qty * factors[unit]

df_influencers["time_spent"] = (
    df_influencers["time_spent"]
    .apply(time_spent_to_days)
    .astype("Int64")
    .fillna(0) + 1
)

In [ ]:
# =============================================================================
# CÁLCULO DO SCORE DE "DISCOVERABILITY" DOS POSTS (0 a 100)
# =============================================================================
# Objetivo académico: Criar um proxy numérico transparente e explicável que estime 
# a "eficiência de descoberta" de um post no feed do LinkedIn, ou seja, quão provável 
# é que o algoritmo o promova a mais pessoas (impressões, alcance orgânico).
#
# Importante: Este NÃO é o algoritmo real do LinkedIn (proprietário e opaco).
# É uma simulação académica baseada em heurísticas conhecidas da literatura e 
# observações empíricas sobre o que o algoritmo valoriza em 2025/2026:
#   - Interações de qualidade (comments > reactions)
#   - Normalização pelo tamanho da audiência (evitar favorecer só contas grandes)
#   - Preferência por formatos ricos (vídeo > imagem > texto)
#   - Efeito moderado de hashtags (com saturação para evitar sinal de spam)
#   - Normalização temporal (engajamento por dia, para comparar posts de idades diferentes)
#
# Esta fórmula serve como target supervisionado para o modelo de ML.
# O facto de o modelo simples dar R² ~0.40 e subir após otimização é esperado e positivo:
#   → Mostra que o XGBoost aprende padrões não-lineares que a heurística manual não captura.
#   → Valida a utilidade de ML sobre regras fixas.
# =============================================================================

# 1. Interação base: ponderação que reflete "qualidade" do engajamento
#    - Comments valem mais (×4) porque indicam discussão profunda (LinkedIn valoriza isso)
#    - Reactions são mais superficiais, mas ainda contam
interaction = (
    1.0 * df_influencers["reactions"].fillna(0) +     # peso 1
    4.0 * df_influencers["comments"].fillna(0)        # peso 4 → mais valioso
)

# 2. Ajuste pelo alcance real (normalização pela audiência)
#    - Divide pelo log1p(followers + 1) para evitar que contas com milhões de seguidores 
#      tenham scores inflacionados só por tamanho da rede
#    - log1p evita divisão por zero e comprime escala (efeito diminishing returns)
reach_adj = interaction / np.log1p(
    df_influencers["followers"].fillna(0).clip(lower=0) + 1
)

# 3. Fator de qualidade contextual (multiplicativo)
#    - media_factor: LinkedIn prioriza formatos visuais/interativos (vídeo > imagem > texto)
media_weight_map = {
    "video":    1.15,     # maior boost (algoritmo promove vídeo)
    "document": 1.10,
    "image":    1.05,
    "text":     1.00,     # baseline
    "article":  0.98,     # ligeira penalização (menos viral)
    "unknown":  1.00      # neutro
}
media_factor = df_influencers["media_type"].str.lower().map(media_weight_map).fillna(1.00)

#    - hashtag_factor: pequeno bónus com saturação (log) para evitar spam
#      coef 0.08 = aumento máximo ~ +30–40% com muitas hashtags
hashtag_factor = 1 + 0.08 * np.log1p(
    df_influencers["num_hashtags"].fillna(0).clip(lower=0)
)

#    - audience_factor: alcance potencial das hashtags (muito suave)
audience_factor = 1 + 0.05 * np.log1p(
    df_influencers["hashtag_followers"].fillna(0).clip(lower=0)
)

# Combinação multiplicativa: cada fator ajusta o alcance ajustado
quality_adj = reach_adj * media_factor * hashtag_factor * audience_factor

# 4. Normalização temporal: engajamento por dia
#    - Divide pelo tempo decorrido (em dias) para comparar posts antigos e recentes de forma justa
#    - clip(lower=1) evita divisão por zero ou valores muito pequenos
engagement_per_day_v2 = quality_adj / df_influencers["time_spent"].fillna(1).clip(lower=1)

# 5. Robustificação contra outliers extremos (virais atípicos)
#    - Clip ao percentil 99: limita efeito de posts "explosivos" que distorcem a escala
p99 = engagement_per_day_v2.quantile(0.99)
clipped = engagement_per_day_v2.clip(upper=p99)

# 6. Compressão logarítmica + normalização min-max para escala 0–100
#    - log1p comprime a cauda pesada (power-law típica de engajamento)
#    - min-max transforma para % interpretável (0 = pior, 100 = melhor no dataset)
x = np.log1p(clipped)
min_x, max_x = x.min(), x.max()

df_influencers["post_discoverability_score"] = (
    100 * (x - min_x) / (max_x - min_x) if max_x > min_x else 0.0
)

# =============================================================================
# Resumo da fórmula final (para relatório / apresentação):
# post_discoverability_score = 100 × min-max( log1p( clip_p99( 
#     (reactions + 4×comments) / log1p(followers+1) × media_factor × hashtag_factor × audience_factor / time_spent 
# ) ) )
#
# Justificativa académica:
# - Proxy transparente e reproduzível
# - Baseado em heurísticas conhecidas do algoritmo LinkedIn 2025/2026
# - Transformações justificadas por natureza dos dados (skewed, outliers, power-law)
# - R² inicial ~0.40 no modelo simples é esperado → target muito dependente de poucas variáveis
# - Melhoria com otimização (RandomizedSearchCV) mostra valor acrescentado do ML
# =============================================================================

In [ ]:
print("Informação de Perfis :")
display(df_profiles.info())
print("="*70)

print("Informação de Pairs :")
display(df_pairs.info())
print("="*70)

print("Informação de Influencers :")
display(df_influencers.info())

In [ ]:
# Configuração visual
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Blues_r")
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

# ----------------------------------------------------------------------------
# 1. Resumo Estatístico dos Targets
# ----------------------------------------------------------------------------
print("═" * 80)
print(" RESUMO ESTATÍSTICO DOS TARGETS DE DISCOVERABILITY ")
print("═" * 80)

print("\nDiscoverability de Perfis (0–100)")
display(df_profiles['profile_discoverability_score'].describe().round(2))

print("\nDiscoverability de Posts (0–100)")
display(df_influencers['post_discoverability_score'].describe().round(2))

# ----------------------------------------------------------------------------
# 2. Teste de Normalidade + Distribuições Interativas
# ----------------------------------------------------------------------------
_, p_profile = shapiro(df_profiles['profile_discoverability_score'].sample(500, random_state=42))
_, p_post = shapiro(df_influencers['post_discoverability_score'].sample(500, random_state=42))
print(f"Shapiro-Wilk Test (sample 500):")
print(f"  Perfis → p-value = {p_profile:.6f}  (não-normal se < 0.05)")
print(f"  Posts  → p-value = {p_post:.6f}  (não-normal se < 0.05)")

fig_profile = px.histogram(
    df_profiles, 
    x='profile_discoverability_score', 
    nbins=40, 
    marginal='box',
    title='Distribuição Interativa – Discoverability de Perfis',
    labels={'profile_discoverability_score': 'Score (0–100)'}
)
fig_profile.show()

fig_post = px.histogram(
    df_influencers, 
    x='post_discoverability_score', 
    nbins=40, 
    marginal='box',
    title='Distribuição Interativa – Discoverability de Posts',
    labels={'post_discoverability_score': 'Score (0–100)'}
)
fig_post.show()

# ----------------------------------------------------------------------------
# 3. Correlações
# ----------------------------------------------------------------------------
print("\n" + "═" * 80)
print(" CORRELAÇÕES MAIS RELEVANTES COM DISCOVERABILITY ")
print("═" * 80)

corr_cols_profile = [
    'profile_discoverability_score', 'num_skills', 'num_experience',
    'num_education', 'about_length', 'headline_length', 'connections', 'years_experience'
]
corr_profile = df_profiles[corr_cols_profile].corr(numeric_only=True)
plt.figure(figsize=(9,7))
sns.heatmap(corr_profile, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5)
plt.title('Correlações – Discoverability de Perfis')
plt.show()

corr_cols_posts = ['post_discoverability_score', 'followers', 'num_hashtags', 'reactions', 'comments']
corr_cols_posts = [c for c in corr_cols_posts if c in df_influencers.columns]
corr_posts = df_influencers[corr_cols_posts].corr(numeric_only=True)
plt.figure(figsize=(8,6))
sns.heatmap(corr_posts, annot=True, fmt='.2f', cmap='Oranges', linewidths=0.5)
plt.title('Correlações – Discoverability de Posts')
plt.show()

# ----------------------------------------------------------------------------
# 4. Comparação por Grupos + Análise de Viés
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.barplot(
    data=df_profiles,
    x='seniority_level',
    y='profile_discoverability_score',
    order=df_profiles.groupby('seniority_level')['profile_discoverability_score'].mean().sort_values(ascending=False).index,
    ax=axes[0,0],
    palette='Blues_d'
)
axes[0,0].set_title('Discoverability média por Seniority Level')
axes[0,0].tick_params(axis='x', rotation=45)

top_industries = df_profiles.groupby('industry')['profile_discoverability_score'].mean().nlargest(8)
sns.barplot(x=top_industries.values, y=top_industries.index, palette='Blues_d', ax=axes[0,1])
axes[0,1].set_title('Top 8 Indústrias – Maior Discoverability Média')
axes[0,1].set_xlabel('Score médio (0–100)')

sns.barplot(
    data=df_influencers,
    x='media_type',
    y='post_discoverability_score',
    order=df_influencers.groupby('media_type')['post_discoverability_score'].mean().sort_values(ascending=False).index,
    palette='Oranges_d',
    ax=axes[1,0]
)
axes[1,0].set_title('Discoverability média por Tipo de Mídia')
axes[1,0].tick_params(axis='x', rotation=45)

df_influencers['hashtags_bin'] = pd.cut(
    df_influencers['num_hashtags'],
    bins=[-1, 0, 2, 5, 10, 20, 50],
    labels=['0', '1–2', '3–5', '6–10', '11–20', '21+']
)
sns.boxplot(data=df_influencers, x='hashtags_bin', y='post_discoverability_score', palette='Oranges_d', ax=axes[1,1])
axes[1,1].set_title('Discoverability por Quantidade de Hashtags')

plt.tight_layout()
plt.show()

# Análise de viés por indústria
bias_industry = df_profiles.groupby('industry')['profile_discoverability_score'].mean().sort_values(ascending=False)
print("\nAnálise de Viés por Indústria (Top 5 e Bottom 5):")
print(bias_industry.head(5))
print("...")
print(bias_industry.tail(5))



In [ ]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

df_profiles.to_pickle(output_dir / "processed_profiles.pkl")
df_influencers.to_pickle(output_dir / "processed_influencers.pkl")



### Conclusões da Análise Exploratória (EDA) – Discoverability no LinkedIn

#### 1. Distribuições dos Scores (Histogramas Interativos + Boxplots)
- **Perfis**: Distribuição bimodal clara, com picos principais em ~20 (perfis "invisíveis" ou com baixa compatibilidade/rede) e ~70–80 (perfis altamente descobríveis). Mediana elevada (~70), muitos valores altos.  
  → Indica dois regimes distintos: perfis médios-baixos vs. perfis estabelecidos (senior/executive com redes densas).  
  → Alinha com o algoritmo do LinkedIn em 2026: prioriza perfis com autoridade, consistência e rede forte (conexões/experiência dominam).  
  → Implicação: Existe desigualdade algorítmica — perfis juniores ou de nichos menos conectados têm menor visibilidade natural.

- **Posts**: Distribuição fortemente enviesada à esquerda (power-law/heavy-tail): maioria dos posts perto de 0 (alcance mínimo), cauda longa com poucos virais até 100. Mediana muito baixa (~0–5), outliers altos.  
  → Padrão clássico de engajamento em redes sociais: 90%+ dos posts têm alcance orgânico quase nulo; poucos explodem.  
  → As transformações (log1p + clip p99 + min-max) foram essenciais para evitar que virais "esmagassem" a escala e para preservar variação útil.  
  → Implicação: Prever discoverability de posts é mais difícil que de perfis → explica R² inicial mais baixo (~0.40) e necessidade de treino rigoroso.

#### 2. Correlações (Heatmaps)
- **Perfis**:  
  - Drivers principais: connections (r=0.68), num_experience (r=0.68), years_experience (r=0.61).  
  - num_skills: r≈0.00 (quase nulo) → quantidade isolada não basta; qualidade/relevância (a explorar via embeddings) será chave.  
  - about_length e headline_length: impacto marginal.  
  → Rede e experiência profissional dominam → confirma que o algoritmo valoriza "autoridade acumulada" e redes densas.

- **Posts**:  
  - Principais preditores: followers (r=0.42), reactions (r=0.42), comments (r=0.34).  
  - num_hashtags: r=0.02 (quase irrelevante) → quantidade excessiva pode sinalizar spam.  
  - Alta correlação interna reactions ↔ comments (r=0.82) → consistência nos dados de interação.  
  → Engajamento direto e tamanho da audiência explicam ~40% da variância → justifica R² moderado inicial; embeddings textuais e interações não-lineares devem melhorar isso.

#### 3. Comparação por Grupos (Barplots e Boxplot)
- **Seniority Level**: Senior (~75) e Executive (~70) muito acima de Entry-level (~15–20) → fator 3–5× superior.  
  → Desigualdade clara: algoritmo favorece perfis com experiência e autoridade.

- **Indústrias (Top 8)**: Energy, Manufacturing, Finance, Telecommunications lideram (scores médios >50). Tecnologia e E-commerce no meio-baixo.  
  → Possível viés: indústrias tradicionais têm redes mais densas e perfis padronizados → maior compatibilidade média.

- **Tipo de Mídia**: Vídeo (~13) e Poll lideram; texto puro, artigo e view no fundo (~2–6).  
  → Confirma prioridade algorítmica por formatos interativos/ricos (LinkedIn 2026 promove vídeo e interatividade).

- **Quantidade de Hashtags**: Pico ideal em 3–10; queda acentuada acima de 11 (média cai para <10).  
  → Sinal clássico de spam: excesso penaliza discoverability.

#### 4. Implicações Gerais e Próximos Passos
- **Positivos**: Padrões claros e alinhados com conhecimento atual do algoritmo LinkedIn (rede, engajamento de qualidade, formatos ricos, autenticidade). EDA robusta e insights acionáveis.
- **Negativos/Limitações**: R² inicial moderado em posts (~0.40) esperado devido à dependência forte de poucas variáveis óbvias. Distribuição power-law torna previsão desafiadora. Possível viés por seniority/indústria (perfis juniores e certas áreas penalizados).
- **Edge Cases**: Perfis sem pares (score=0), posts virais atípicos (outliers), hashtags excessivas (queda abrupta).
- **Alinhamento com o Trabalho**: Esta EDA cobre bem (b) Data Understanding e inicia (c) Data Preparation. Motiva o uso de embeddings (qualidade textual) e treino rigoroso (não-linearidades) para melhorar o R².
- **Decisão**: Resultados consistentes e ricos → **podemos avançar para featureEngineering** (embeddings + PCA) e modelTraining com confiança. Próximo foco: capturar semântica (texto) e interações complexas para subir o R² dos posts.

Pronto para o relatório e apresentação oral!